In [ ]:
#!/usr/bin/env python3
"""
Compute mutual information between all features and a target class column.

Inputs:
- CSV file with features and a target column named "Severity" by default.

Outputs:
- mutual_information_all.csv
- mutual_information_top60.csv
- mutual_information_top60.png
"""

import argparse
import os
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.feature_selection import mutual_info_classif
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import LabelEncoder


def infer_feature_types(df: pd.DataFrame, target_col: str):
    """Split columns into categorical and numerical."""
    feature_cols = [c for c in df.columns if c != target_col]

    categorical_cols = []
    numerical_cols = []

    for col in feature_cols:
        if pd.api.types.is_numeric_dtype(df[col]):
            numerical_cols.append(col)
        elif pd.api.types.is_bool_dtype(df[col]):
            categorical_cols.append(col)
        else:
            categorical_cols.append(col)

    return categorical_cols, numerical_cols


def prepare_target(y: pd.Series):
    """Encode target labels to integers and drop missing target values."""
    y = y.copy()
    mask = ~y.isna()
    y = y.loc[mask]

    le = LabelEncoder()
    y_encoded = le.fit_transform(y.astype(str))
    return y_encoded, mask, le


def compute_mutual_information(df: pd.DataFrame, target_col: str = "Severity", random_state: int = 42):
    if target_col not in df.columns:
        raise ValueError(f"Target column '{target_col}' not found in CSV.")

    # Keep only rows where target exists
    df = df.loc[~df[target_col].isna()].copy()

    y = df[target_col].astype(str)
    le_y = LabelEncoder()
    y_encoded = le_y.fit_transform(y)

    results = []

    for col in df.columns:
        if col == target_col:
            continue

        # Use only rows where this feature is present
        mask = ~df[col].isna()
        if mask.sum() < 2:
            results.append((col, np.nan))
            continue

        y_col = y_encoded[mask]
        x_col = df.loc[mask, col]

        # Numerical feature
        if pd.api.types.is_numeric_dtype(x_col):
            X = x_col.to_numpy().reshape(-1, 1)
            discrete = False

        # Categorical / string / boolean feature
        else:
            x_as_str = x_col.astype(str)
            le_x = LabelEncoder()
            X = le_x.fit_transform(x_as_str).reshape(-1, 1)
            discrete = True

        try:
            mi = mutual_info_classif(
                X,
                y_col,
                discrete_features=[discrete],
                random_state=random_state
            )[0]
        except Exception:
            mi = np.nan

        results.append((col, mi))

    result_df = pd.DataFrame(results, columns=["feature", "mutual_information"])
    result_df = result_df.sort_values("mutual_information", ascending=False, na_position="last").reset_index(drop=True)
    return result_df


def save_outputs(result_df: pd.DataFrame, df: pd.DataFrame, output_dir: str):
    os.makedirs(output_dir, exist_ok=True)

    full_path = os.path.join(output_dir, "mutual_information_all.csv")
    top60_path = os.path.join(output_dir, "mutual_information_top60.csv")
    png_path = os.path.join(output_dir, "mutual_information_top60.png")

    # Save full ranking
    result_df.to_csv(full_path, index=False)

    # Save top 60
    top60 = result_df.head(60).copy()
    top60.to_csv(top60_path, index=False)

    # Plot top 60 as a horizontal bar chart
    plt.figure(figsize=(12, 8))
    top60_sorted = top60.sort_values("mutual_information", ascending=True)

    plt.barh(top60_sorted["feature"], top60_sorted["mutual_information"])
    plt.xlabel("Mutual Information")
    plt.ylabel("Feature")
    plt.title("Top 60 Features by Mutual Information vs Severity")
    plt.tight_layout()
    plt.savefig(png_path, dpi=200, bbox_inches="tight")
    plt.close()
    # ---- NEW: histogram distributions for top 60 features ----
    hist_path = os.path.join(output_dir, "top60_feature_distributions.png")

    top_features = result_df.head(60)["feature"].tolist()

    n_cols = 4
    n_rows = int(np.ceil(len(top_features) / n_cols))

    plt.figure(figsize=(20, 5 * n_rows))

    for i, feature in enumerate(top_features, 1):
        plt.subplot(n_rows, n_cols, i)

        if feature not in df.columns:
            continue

        data = df[feature].dropna()

        if data.empty:
            continue

        # Numerical
        if pd.api.types.is_numeric_dtype(data):
            plt.hist(data, bins=30)
        else:
            # Categorical → top 20 categories
            vc = data.astype(str).value_counts().head(20)
            plt.barh(vc.index, vc.values)

        plt.title(feature)

    plt.tight_layout()
    plt.savefig(hist_path, dpi=200, bbox_inches="tight")
    plt.close()

    return full_path, top60_path, png_path


def main():
    csv_file = "final_processed_with_interest.csv"
    target_col = "Severity"
    output_dir = "/content/mi_output"
    random_state = 42

    try:
        df = pd.read_csv(csv_file)
        cols_to_drop = ["requested_datetime_utc", "returned_time_utc", "generationtime_ms", "weathercode", "closest_image"]
        df = df.drop(columns=[c for c in cols_to_drop if c in df.columns])
    except Exception as e:
        print(f"Error reading CSV: {e}", file=sys.stderr)
        sys.exit(1)

    try:
        result_df = compute_mutual_information(
          df=df,
          target_col=target_col,
          random_state=random_state,
        )
        full_path, top60_path, png_path = save_outputs(result_df, df, output_dir)

        print("Done.")
        print(f"Full ranking: {full_path}")
        print(f"Top 60 CSV:   {top60_path}")
        print(f"Top 60 PNG:   {png_path}")
        print("\nTop 60 features:")
        print(result_df.head(60).to_string(index=False))

    except Exception as e:
        print(f"Error: {e}", file=sys.stderr)
        sys.exit(1)


if __name__ == "__main__":
    main()

In [ ]:
# === CONFIG ===
csv_file = "/content/final_processed_with_interest.csv"
top_features_file = "/content/mi_output/mutual_information_top60.csv"
output_dir = "/content/mi_output/distributions_slides"

features_per_slide = 12  # 12 = nice 3x4 layout
dpi = 300  # high resolution for Slides

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

os.makedirs(output_dir, exist_ok=True)

# === LOAD DATA ===
df = pd.read_csv(csv_file)

cols_to_drop = ["requested_datetime_utc", "returned_time_utc", "generationtime_ms", "weathercode", "closest_image"]
df = df.drop(columns=[c for c in cols_to_drop if c in df.columns])

top_df = pd.read_csv(top_features_file)
top_features = top_df["feature"].tolist()

# === SPLIT INTO CHUNKS ===
def chunks(lst, n):
    for i in range(0, len(lst), n):
        yield lst[i:i + n]

feature_chunks = list(chunks(top_features, features_per_slide))

# === GENERATE SLIDES ===
for idx, feature_group in enumerate(feature_chunks, 1):
    n = len(feature_group)

    # Layout: 3 rows x 4 cols (landscape-friendly)
    n_cols = 4
    n_rows = int(np.ceil(n / n_cols))

    plt.figure(figsize=(16, 9))  # LANDSCAPE

    for i, feature in enumerate(feature_group, 1):
        plt.subplot(n_rows, n_cols, i)

        if feature not in df.columns:
            continue

        data = df[feature].dropna()

        if data.empty:
            continue

        # Numeric
        if pd.api.types.is_numeric_dtype(data):
            plt.hist(data, bins=30)
        else:
            vc = data.astype(str).value_counts().head(15)
            plt.barh(vc.index, vc.values)

        plt.title(feature, fontsize=10)
        plt.xticks(fontsize=8)
        plt.yticks(fontsize=8)

    plt.tight_layout()

    out_path = os.path.join(output_dir, f"feature_distributions_slide_{idx}.png")
    plt.savefig(out_path, dpi=dpi, bbox_inches="tight")
    plt.close()

    print(f"Saved: {out_path}")

In [ ]:

import pandas as pd
# Get the uploaded file name
file_name = "final_processed_with_interest.csv"

# Load CSV
df = pd.read_csv(file_name)

# Check column exists
if "Severity" not in df.columns:
    raise ValueError("CSV does not contain a 'Severity' column.")

# Normalize values
df["Severity"] = df["Severity"].astype(str).str.strip().str.capitalize()

# Count values
counts = df["Severity"].value_counts()

total_rows = len(df)

for category in ["Slight", "Fatal", "Serious"]:
    count = counts.get(category, 0)
    pct = (count / total_rows) * 100
    print(f"{category}: {pct:.2f}% ({count} rows)")

print(f"\nTotal rows: {total_rows}")
print("Number of columns:", len(df.columns))